# Business Entity Resolution — Kaggle accelerator notebook

Runs the `code/business_entity_resolution` pipeline on Kaggle, using the GPU where it
genuinely helps and 30 GB RAM to train on far more data than a 16 GB laptop can.

## Before you run
1. Upload the competition data as a **private Dataset** — the folder that contains
   `train/` and `test/` (i.e. `.../student_resource/dataset`).
2. Upload the code folder as a second **private Dataset** — the folder that contains
   `src/` (i.e. `code/business_entity_resolution`).
3. Notebook settings → **Accelerator: GPU T4 x2** (or P100); **Internet: On**;
   **Persistence: Files only**.
4. Run All. Outputs land in `/kaggle/working/output/`.

## Honest scope (read this)
- **Only two things can use the GPU here**: LightGBM (OpenCL backend, *if* the image's
  build has it) and an optional neural reranker (final section).
- **The dominant costs are CPU-bound** — rapidfuzz pair features and sparse retrieval.
  Kaggle GPU sessions have ~4 vCPU, so expect each stage to be *slower* than on a
  16-thread desktop.
- **The real win is 30 GB RAM**: `build_training_set --sample 500000` (2.5x the laptop
  run) fits comfortably and gives a materially better classifier.
- **Blocking caps the metric at ~0.98** (measured oracle 0.9830 at K=100). No notebook,
  GPU or otherwise, reaches 0.99 from this design. See `docs/` §3.5/§3.6.

## Time budget (important)
Kaggle GPU sessions get ~4 vCPU. Rough estimates on this pipeline: train index ~40 min,
test index ~40 min, 500K training set ~1–1.5 h, training ~30 min, **test inference 4–6 h**.
Total ≈ 7–9 h against a 12 h session limit. If you are short on time, lower `SAMPLE`
(accuracy cost) and/or `KMAX`. Caches live in `/kaggle/temp`, which is **not** persisted,
so a session must run to completion — prefer *Save & Run All* and don't rely on resuming.


In [ ]:
# --- knobs ---
SAMPLE = 500_000   # train S1 sample (laptop used 200k). Lower = faster, slightly worse.
KMAX   = 50        # candidates retrieved per S1
BATCH  = 20_000    # queries per inference batch
NJOBS  = 4         # ~4 vCPU on Kaggle GPU sessions


In [ ]:
import glob, importlib, os, platform, shutil, subprocess, sys

print('python', platform.python_version())
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or gpu.stderr.strip() or 'none')
try:
    import psutil; print('RAM GB:', round(psutil.virtual_memory().total / 1e9, 1))
except Exception as e:
    print('psutil unavailable:', e)


In [ ]:
# --- install anything the image is missing (needs Internet: On) ---
MODULES = {'pandas': 'pandas', 'numpy': 'numpy', 'scipy': 'scipy',
           'sklearn': 'scikit-learn', 'lightgbm': 'lightgbm', 'rapidfuzz': 'rapidfuzz',
           'sparse_dot_topn': 'sparse-dot-topn', 'anyascii': 'anyascii',
           'pyarrow': 'pyarrow'}
missing = []
for mod, pkg in MODULES.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(pkg)
if missing:
    print('installing:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=False)
for mod in MODULES:
    try:
        importlib.import_module(mod); print('ok  ', mod)
    except Exception as e:
        print('FAIL', mod, e)


In [ ]:
# --- locate the data + code datasets (paths vary by upload structure) ---
def find_first(pattern, roots=('/kaggle/input', '/kaggle/working')):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', pattern), recursive=True))
        if hits:
            return hits[0]
    return None

s1 = find_first('test_source1.tsv')
assert s1, 'test_source1.tsv not found under /kaggle/input — add the data dataset'
DATA_DIR = os.path.dirname(os.path.dirname(s1))          # .../dataset
assert os.path.isdir(os.path.join(DATA_DIR, 'train')), DATA_DIR

src_pkg = find_first('text_cleaner.py')
assert src_pkg, 'src/ not found under /kaggle/input — add the code dataset'
CODE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(src_pkg)))  # .../business_entity_resolution
assert os.path.isdir(os.path.join(CODE_DIR, 'src')), CODE_DIR

print('DATA_DIR =', DATA_DIR)
print('CODE_DIR =', CODE_DIR)
print('test files:', sorted(os.listdir(os.path.join(DATA_DIR, 'test'))))


In [ ]:
# --- environment for every stage ---
WORK = '/kaggle/temp/ber_work' if os.path.isdir('/kaggle/temp') else '/kaggle/working/work'
OUT  = '/kaggle/working/output'        # final TSVs (small; safe for the output limit)
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print('WORK =', WORK)

ENV = dict(os.environ, ER_DATA_DIR=DATA_DIR, ER_WORK_DIR=WORK, ER_OUTPUT_DIR=OUT)

# GPU probe for LightGBM: the pip/Kaggle wheel may or may not be GPU-enabled.
import numpy as np, lightgbm as lgb
def lgb_gpu_ok():
    X = np.random.rand(2000, 10).astype(np.float32); y = (X[:, 0] > .5).astype(int)
    try:
        lgb.train({'objective': 'binary', 'device_type': 'gpu', 'verbose': -1,
                   'num_leaves': 15, 'min_data_in_leaf': 5},
                  lgb.Dataset(X, label=y), num_boost_round=3)
        return True
    except Exception as e:
        print('LightGBM GPU unavailable:', str(e)[:120]); return False
LGB_DEVICE = 'gpu' if lgb_gpu_ok() else 'cpu'
ENV['ER_LGB_DEVICE'] = LGB_DEVICE
print('LightGBM device:', LGB_DEVICE)

def run(module, *args):
    """Run a package module (-m src....) or a script (utils/*.py) with cwd=CODE_DIR."""
    head = [sys.executable, module] if module.endswith('.py') else [sys.executable, '-m', module]
    cmd = head + [str(a) for a in args]
    print('>>', ' '.join(cmd), flush=True)
    subprocess.run(cmd, cwd=CODE_DIR, env=ENV, check=True)

print('ready')


## Stage 1 — blocking (hashed sparse index + measured recall@K)

Builds the S2/S3 index for a corpus. Every stage is idempotent (skip-if-exists),
so re-running a cell after an interruption is cheap.


In [ ]:
run('src.blocking.build_index', '--corpus', 'train')   # ~10.3M docs; slowest stage on 4 cores


In [ ]:
run('src.blocking.evaluate_blocking', '--sample', 40000, '--kmax', 100)
print('^ recall@K + ORACLE ceiling: this is the hard upper bound for ANY classifier.')


In [ ]:
run('src.blocking.build_index', '--corpus', 'test')


## Stage 2 — training set + LightGBM

`--sample 500000` trains on 2.5x the entities the laptop used (30 GB RAM makes it fit).
Negatives come from real blocking output, labels from ground truth.


In [ ]:
run('src.matching.build_training_set', '--sample', SAMPLE, '--n-jobs', NJOBS)


In [ ]:
run('src.matching.train_matcher', '--device', LGB_DEVICE, '--n-jobs', NJOBS)
print('^ prints oracle / all-empty / top-1 baselines and the tuned selection rule.')


## Stage 3 — test inference + validation

Per-country / per-batch, checkpointed: if the session dies mid-way, re-running resumes
the countries already finished.


In [ ]:
run('src.matching.predict_test', '--kmax', KMAX, '--batch', BATCH, '--n-jobs', NJOBS)


In [ ]:
run('utils/validate_submission.py',
    '--matching', os.path.join(OUT, 'matching_results.tsv'),
    '--candidate', os.path.join(OUT, 'candidate_pairs.tsv'),
    '--test-dir', os.path.join(DATA_DIR, 'test'))

import csv
from collections import Counter
mm, n = Counter(), 0
with open(os.path.join(OUT, 'matching_results.tsv')) as f:
    rd = csv.reader(f, delimiter='\t'); next(rd)
    for _sid, m in rd:
        n += 1; mm[m.count(',') + 1 if m else 0] += 1
print('rows', n, '| predicted-empty %.2f%%' % (100 * mm[0] / n))

sub = '/kaggle/working/submission'; os.makedirs(sub, exist_ok=True)
for fn in ('matching_results.tsv', 'candidate_pairs.tsv'):
    shutil.copy(os.path.join(OUT, fn), os.path.join(sub, fn))
print('copied to', sub, '-> upload matching_results.tsv to the leaderboard')


## Optional — GPU neural reranker (off by default, untested)

This is the only stage with real GPU headroom: a cross-encoder that re-scores the
LightGBM's top candidates. The plan (§8.3) ranks it the main way to close the ~4-pt
classifier gap, and it fits an 8–16 GB GPU with short pairs. It is **not** needed for a
valid submission — the classical outputs above are already produced and validated.

Set `RUN_RERANKER = True` to attempt it; it needs `transformers` + `torch` (Internet)
and a cached pair file. Treat the code below as a starting point, not a finished model.


In [ ]:
RUN_RERANKER = False   # flip to True to experiment (needs Internet: On)

if RUN_RERANKER:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch'],
                   check=False)
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
    # Minimal sketch: score (S1, candidate) text pairs with a pretrained cross-encoder.
    CKPT = os.path.join(WORK, 'cache', 'train_set')
    X = np.load(os.path.join(CKPT, 'X.npy'), mmap_mode='r')
    print('cached pairs:', X.shape, '-> fine-tune a cross-encoder on these for a real gain')
    # tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-xsmall')
    # model = AutoModelForSequenceClassification.from_pretrained(...).cuda().half()
    #   train on (name1+addr1, name2+addr2) pairs with the cached labels, then blend
    #   reranker_score as an extra LightGBM feature (or override selection).
    # See docs/FINAL_RESEARCH_AND_IMPLEMENTATION_PLAN.md §8.3.
else:
    print('reranker skipped (classical outputs are complete and validated)')
